<a href="https://colab.research.google.com/github/bfs0/SDAIA_Athkax_A-B_Testing_PROJECT/blob/main/Copy_of_%D8%AA%D9%85%D8%B1%D9%8A%D9%86_1_2%D8%8C_%D8%A7%D9%84%D8%AA%D9%88%D8%B2%D9%8A%D8%B9_%D8%A7%D9%84%D8%AB%D8%A7%D8%A8%D8%AA_%D9%81%D9%8A_%D8%A7%D9%84%D8%A5%D8%B7%D9%84%D8%A7%D9%82_%D8%A7%D9%84%D8%AA%D8%AF%D8%B1%D9%8A%D8%AC%D9%8A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# تمرين 1.2: التوزيع الثابت في الإطلاق التدريجي

**برنامج التجارب واختبارات A/B والاستدلال السببي**، اليوم الأول

**الخطوات:**
1. من قائمة File اختر Save a copy in Drive، واعمل على النسخة التي تُفتح.
2. شغّل الخلايا بالترتيب، ثم اكتب حلك في الخلية المخصصة.
3. اضغط Share، واجعل General access على Anyone with the link بصلاحية Viewer، ثم اضغط Copy link.
4. الصق الرابط في ملف التقييم الخاص بمجموعتك، في عمود «تمرين 1.2».

</div>

<div dir="rtl">

## الإعداد

شغّل الخلية التالية أولًا.

</div>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))

import hashlib

say("الإعداد جاهز")

<div dir="rtl">

## الأدوات الجاهزة

هذه الدوال والبيانات من دفتر اليوم، ويحتاجها التمرين. شغّلها كما هي.

</div>

In [2]:
def assign_variant(user_id, experiment="checkout_v2", treatment_share=0.5):
    """توزيع ثابت: المستخدم نفسه يحصل على النسخة نفسها دائمًا في التجربة نفسها."""
    digest = hashlib.sha256(f"{experiment}:{user_id}".encode()).hexdigest()
    bucket = int(digest[:8], 16) / 0xFFFFFFFF          # رقم بين 0 و1
    return "B" if bucket < treatment_share else "A"


say("نسخة user_123 في خمس محاولات:", [assign_variant("user_123") for _ in range(5)])

users = [f"user_{i}" for i in range(100_000)]
checkout = pd.Series([assign_variant(u, "checkout_v2") for u in users], name="checkout_v2")
banner = pd.Series([assign_variant(u, "home_banner") for u in users], name="home_banner")
print(checkout.value_counts(normalize=True).round(4))
pd.crosstab(checkout, banner, normalize=True).round(3)

checkout_v2
A    0.5024
B    0.4976
Name: proportion, dtype: float64


home_banner,A,B
checkout_v2,,
A,0.251,0.251
B,0.248,0.249


<div dir="rtl">

## المطلوب

في الإطلاق التدريجي نبدأ بنسبة 10% في النسخة B ثم نرفعها إلى 50%.
1. احسب نسبة B الفعلية عند `treatment_share=0.10`.
2. تأكد أن كل من كان في B عند 10% **بقي في B** عند 50%. لماذا هذا مهم؟

</div>

In [3]:
# تلميح: كوّن مجموعة (set) لمستخدمي B عند 0.10 وأخرى عند 0.50، ثم قارن بينهما بالعلامة <=
b10 = set(u for u in users if assign_variant(u, "checkout_v2", 0.10) == "B")
b50 = set(u for u in users if assign_variant(u, "checkout_v2", 0.50) == "B")


# المقارنه
print(b10 <= b50)
# عشان المستخدم اللي جرب النسخة الجديدة ما تضيع عليه وتختفي لما نرفع النسبة وتتأثر النتائج





True


<div dir="rtl">

## التسليم

انتهيت؟ اضغط Share، ثم Copy link، والصق الرابط في ملف التقييم في عمود «تمرين 1.2».

</div>